# ⚖️ Veredito B2 — equipamentos que desgastam, quebram e param deixam a crônica mais viva?

Notebook de veredito da **Fase 4 do Degrau 1** (B2 Equipamentos: idade/condição por instalação, capacidade efetiva `capacity·h·(1−0,15·min(1, age/240))`, desgaste por utilização, hazard de quebra de Weibull β≈1,6 calibrado nos benchmarks de frotas, reparo **decidido pela política** a 8–15% do custo, manutenção preventiva a ~2,2% restaurando h, overhaul a 60% zerando idade e h). Protocolo **pré-registrado**:

- Corridas **pareadas**: seeds 42, 7, 11, 23 × (B2 desligado) × (B2 ligado), 8 empresas, 120 ticks — só a flag `equip_v2` muda. A base é o mundo vivo completo (B1 mercado vivo + B3 país/política + B4a importações ligados), porque é nesse mundo que o B2 vai viver se as fases anteriores forem adotadas.
- Métricas: eventos salientes da crônica por corrida (sem `flows`/`founding`/`end`), quebras (`breakdown`), eventos de manutenção por ação (`maintenance` com `data.action` em repair/preventive/overhaul), disponibilidade da frota e por arquétipo (1 − meses parada / meses de instalação, de `result()["equipment"]`), falências e causas (incl. a causa nova `quebra`), sinks de equipamento nos flows (spend_repair/maintenance/overhaul).
- **Regra de aprovação (régua do PRD + portão da Fase 4 no design §5):** (1) ledger fecha em TODAS as corridas — dinheiro incluindo a conta externo e massa incluindo a fonte `imported`; **E** (2) eventos salientes ON > OFF em ≥ 3 de 4 seeds; **E** (3) ≥ 3 quebras por corrida ON em ≥ 3 de 4 seeds (a frota visivelmente viva, não um enfeite); **E** (4) disponibilidade média da frota ON dentro de [0,78; 0,95] em ≥ 3 de 4 seeds (a faixa do design: 80–85% rodando até quebrar, 92–95% mantendo — a frota mista mora entre os dois); **E** (5) nenhum arquétipo falido em TODAS as corridas ON (quebra não pode ser sentença de morte de um estilo); **E** (6) ≥ 4 empresas vivas ao fim de cada corrida ON. Fora isso: REPROVADO/AJUSTAR.

Regra do fundador: simulação de verdade roda AQUI, no Kaggle — nunca na VM de desenvolvimento.


In [ ]:
# 1) Fonte do pacote: dataset privado do bundle mais recente
#    (mundo-autonomo-srcN — criado a partir do repo com
#    scripts/kaggle_push_kernel.py --bundle-src no lançamento).
#    O pacote e puro Python: NAO precisa de pip install. Basta src/ no path.
#    LOADER RECURSIVO (26/10/2026): o Kaggle mudou o layout do mount de
#    datasets — agora eles aparecem em /kaggle/input/datasets/<dono>/<slug>/
#    com a arvore JA EXTRAIDA (sem o zip dentro), e o caminho classico
#    /kaggle/input/<slug>/ nao existe mais. A busca aceita os DOIS layouts:
#    qualquer 'src/mundo/__init__.py' sob /kaggle/input, com fallback
#    para o zip mundo-src.zip se ele existir.
import glob, os, sys, zipfile

inits = sorted(glob.glob('/kaggle/input/**/src/mundo/__init__.py',
                         recursive=True))
zip_cands = sorted(glob.glob('/kaggle/input/**/mundo*.zip', recursive=True))
if inits:
    src_root = os.path.dirname(os.path.dirname(os.path.dirname(inits[-1])))
elif zip_cands:
    zipfile.ZipFile(zip_cands[-1]).extractall('/kaggle/working/pkg')
    src_root = '/kaggle/working/pkg'
else:
    raise SystemExit('Fonte do pacote nao encontrada (loader recursivo): '
                     'anexe o dataset pascalcoimbra/mundo-autonomo-srcN '
                     'a este kernel.')
print('pacote em', src_root)
sys.path.insert(0, os.path.join(src_root, 'src'))
import mundo
print('mundo', mundo.__version__, 'importado de', src_root)


## As corridas pareadas
Cada corrida imprime o placar no objeto de resultados; a conservação é verificada corrida a corrida a partir dos contadores do motor — dinheiro **incluindo a conta externo** do B4a e massa **incluindo a fonte `imported`**, como na Fase 3. Os sinks de equipamento (reparo/PM/overhaul) são gastos ao "mundo" via `money_out`, então o ledger de dinheiro fecha sem termo novo.


In [ ]:
# 2) Corridas pareadas: MESMAS seeds, mundo vivo completo x + B2.
import statistics

from mundo import chain
from mundo.engine import Engine

SEEDS = [42, 7, 11, 23]
TICKS = 120
SALIENT_IGNORED = {"flows", "founding", "end"}


def metrics(eng, res):
    kinds = {}
    maint_actions = {}
    causes = {}
    sinks = {"spend_repair": 0.0, "spend_maintenance": 0.0,
             "spend_overhaul": 0.0}
    for e in res["chronicle"]:
        kinds[e["kind"]] = kinds.get(e["kind"], 0) + 1
        if e["kind"] == "maintenance":
            a = e["data"].get("action")
            maint_actions[a] = maint_actions.get(a, 0) + 1
        if e["kind"] == "flows":
            for rec in e["data"]["companies"].values():
                for k in sinks:
                    sinks[k] += -rec["flows"].get(k, 0.0)
        if e["kind"] == "bankruptcy":
            causes[e["data"]["cause"]] = causes.get(e["data"]["cause"], 0) + 1
    salient = sum(v for k, v in kinds.items() if k not in SALIENT_IGNORED)
    total = sum(c.cash for c in eng.companies)
    money_ok = abs(total - (eng.initial_cash + eng.money_in
                            + eng.money_from_external - eng.money_out
                            - eng.money_to_external)) < 1e-4
    mass_ok = True
    for c in chain.COMMODITIES:
        led = eng.ledger_mass[c]
        stocks = sum(comp.stocks[c] for comp in eng.companies)
        bal = led["extracted"] + led["produced"] \
            + led.get("imported", 0.0) - led["consumed"] \
            - led["sold_external"]
        if abs(stocks - bal) > 1e-3:
            mass_ok = False
    eq = res.get("equipment")
    by_arch = {}
    if eq:
        for st in eq["companies"].values():
            by_arch[st["archetype"]] = st["availability"]
    top = res["scoreboard"][0]
    return {
        "salient": salient,
        "breakdowns": kinds.get("breakdown", 0),
        "maintenance_events": maint_actions,
        "bankruptcies": kinds.get("bankruptcy", 0),
        "bankruptcy_causes": causes,
        "dead_archetypes": [c.archetype for c in eng.companies
                            if not c.alive],
        "sinks": sinks,
        "fleet_availability": (eq["fleet"]["availability"]
                               if eq else None),
        "availability_by_archetype": by_arch,
        "ledger_ok": money_ok and mass_ok,
        "top_cash_name": top["name"], "top_cash": top["cash"],
        "alive": sum(1 for r in res["scoreboard"] if r["alive"]),
        "supercomputers": int(eng.total_supercomputers),
    }


results = {}
for seed in SEEDS:
    for label, on in (("off", False), ("on", True)):
        eng = Engine(seed=seed, n_companies=8, market_v2=True,
                     country_v2=True, equip_v2=on)
        res = eng.run(TICKS)
        results[(seed, label)] = metrics(eng, res)
        print(f"seed {seed} B2 {label}: pronto")


## O veredito
A tabela compara OFF × ON por seed e a última linha imprime o veredito pela regra pré-registrada. As quebras sozinhas não aprovam nada: a régua exige a crônica mais viva no total, a disponibilidade da frota dentro da faixa do design (nem frota imortal, nem cemitério de plantas), nenhum arquétipo condenado pela quebra e os ledgers fechando com os sinks novos dentro do `money_out` de sempre.


In [ ]:
# 3) Comparativo e VEREDITO (regra pre-registrada no topo do notebook)
import json

print(f"{'seed':>5} {'metrica':<26} {'B2 off':>10} {'B2 on':>10}")
wins_salient = 0
breaks_ok = 0
avail_ok = 0
ledger_all = True
alive_ok = True
dead_by_arch = {}
for seed in SEEDS:
    off, on = results[(seed, "off")], results[(seed, "on")]
    ledger_all = ledger_all and off["ledger_ok"] and on["ledger_ok"]
    alive_ok = alive_ok and on["alive"] >= 4
    if on["salient"] > off["salient"]:
        wins_salient += 1
    if on["breakdowns"] >= 3:
        breaks_ok += 1
    if on["fleet_availability"] is not None \
            and 0.78 <= on["fleet_availability"] <= 0.95:
        avail_ok += 1
    for a in on["dead_archetypes"]:
        dead_by_arch[a] = dead_by_arch.get(a, 0) + 1
    for key in ("salient", "breakdowns", "bankruptcies"):
        print(f"{seed:>5} {key:<26} {off[key]:>10.0f} {on[key]:>10.0f}")
    print(f"{seed:>5} {'disponibilidade frota':<26} {'-':>10} "
          f"{on['fleet_availability']:>10.3f}")
    print(f"{seed:>5} {'sinks equip. US$ (on)':<26} {'-':>10} "
          f"{sum(on['sinks'].values()):>10.0f}")
    print(f"{seed:>5} {'manutencao por acao (on)':<26} "
          f"{str(on['maintenance_events'])[:44]:>54}")
    print(f"{seed:>5} {'causas de falencia (on)':<26} "
          f"{str(on['bankruptcy_causes'])[:44]:>54}")
    print(f"{seed:>5} {'placar: lider de caixa':<26} "
          f"{off['top_cash_name'][:10]:>10} {on['top_cash_name'][:10]:>10}")
arch_doomed = [a for a, n in dead_by_arch.items() if n >= len(SEEDS)]
print()
print(f"eventos salientes: B2 on vence em {wins_salient}/{len(SEEDS)} seeds")
print(f"corridas on com >= 3 quebras: {breaks_ok}/{len(SEEDS)}")
print(f"corridas on com disponibilidade em [0,78; 0,95]: "
      f"{avail_ok}/{len(SEEDS)}")
print(f"ledger fecha (dinheiro c/ externo + massa c/ imported) "
      f"em todas: {ledger_all}")
print(f"corridas on com >= 4 empresas vivas: {alive_ok}")
print(f"arquetipos falidos em TODAS as corridas on: {arch_doomed}")

approved = (ledger_all and alive_ok and wins_salient >= 3
            and breaks_ok >= 3 and avail_ok >= 3 and not arch_doomed)
print()
print("VEREDITO B2:", "APROVADO — a frota desgasta, quebra, para e "
      "volta, a cronica fica visivelmente mais viva e os ledgers fecham "
      "com os sinks de equipamento." if approved else
      "REPROVADO/AJUSTAR — ver metricas acima: a regua do PRD nao foi "
      "atingida; calibrar parametros (eta/wear/reparo) antes de "
      "qualquer nova tentativa.")
with open("/kaggle/working/veredito-b2.json", "w", encoding="utf-8") as f:
    json.dump({f"{s}-{l}": m for (s, l), m in results.items()}, f,
              ensure_ascii=False, indent=1)
print("detalhes em /kaggle/working/veredito-b2.json")
